
# Processing Silver layer for Account_move model

Process for create a silver layer for the account_move model, for build that I created a temp view, this temp view has some features like compute the following fields:
- company_id: array
- journal_id: array
- partner_id: String, extract the information from the field, id and name
- cast all date fields
- Quility row_over() for get only the last one record created.

and get the merge code by account_move_id_ for avoid the duplicate information.

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stg_account_move AS
SELECT 
  id AS move_id,
  move_type,
  name,
  payment_reference,
  state,
  ref,
  CAST(create_date AS TIMESTAMP) AS created_at,
  CAST(write_date AS TIMESTAMP) AS updated_at,
  processing_timestamp AS bronze_ingested_at
FROM bakery_pos_develop.bronze.account_move
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY id 
    ORDER BY write_date DESC, processing_timestamp DESC
) = 1

### Is fundamental to create the table with the structure that coincidence with 100% with the view, because generate an error multiple times.


In [0]:
%sql

CREATE TABLE IF NOT EXISTS bakery_pos_develop.silver.account_move (
  move_id INT,
  move_type STRING,
  name STRING,
  payment_reference STRING,
  state STRING,
  ref STRING,
  created_at TIMESTAMP,
  updated_at TIMESTAMP,
  bronze_ingested_at TIMESTAMP,
  silver_processed_at TIMESTAMP,
  is_updated BOOLEAN,
  is_deleted BOOLEAN
) USING DELTA;


In [0]:

%sql

MERGE INTO bakery_pos_develop.silver.account_move AS target
USING (
  SELECT *, current_timestamp() AS silver_processed_at 
  FROM stg_account_move
) AS source
ON target.move_id = source.move_id
WHEN MATCHED AND source.updated_at > target.updated_at THEN
  UPDATE SET
    name = source.name,
    ref = source.ref,
    move_type = source.move_type,
    state = source.state,
    payment_reference = source.payment_reference,
    created_at = source.created_at,
    updated_at = source.updated_at,
    is_updated = True,
    is_deleted= False,
    bronze_ingested_at = source.bronze_ingested_at,
    silver_processed_at = source.silver_processed_at
WHEN NOT MATCHED THEN 
  INSERT (
    move_id,
    move_type,
    name,
    payment_reference,
    state,
    ref,
    created_at,
    updated_at,
    bronze_ingested_at,
    silver_processed_at,
    is_updated,
    is_deleted
  )
  VALUES (
    source.move_id,
    source.move_type,
    source.name,
    source.payment_reference,
    source.state,
    source.ref,
    source.created_at,
    source.updated_at,
    source.bronze_ingested_at,
    source.silver_processed_at,
    False,
    False
  );

In [0]:
import datetime
import pytz

bogota_tz = pytz.timezone("America/Bogota")

year = datetime.datetime.now(tz=bogota_tz).strftime("%Y")
day = datetime.datetime.now(tz=bogota_tz).strftime("%d")
month = datetime.datetime.now(tz=bogota_tz).strftime("%m")

In [0]:
%sql

CREATE TEMP VIEW stg_account_move_deletes AS
SELECT *, current_timestamp() AS silver_processed_at FROM read_files(
  's3://bakery-pos-bk-978430483777-us-east-2-an/odoo_deletes/${year}/${month}/${day}',
  format => 'json',
  multiLine => 'true'
)

In [0]:
%sql

MERGE INTO bakery_pos_develop.silver.account_move AS target
USING stg_account_move_deletes AS source
ON target.account_move_id = source.record_id
WHEN MATCHED THEN
  UPDATE SET
    updated_by = source.user_id,
    state = 'deleted',
    silver_processed_at = source.silver_processed_at;